# Parts-of-Speech Tagging - First Steps: Working with text files, Creating a Vocabulary and Handling Unknown Words

In this lecture notebook you will create a vocabulary from a tagged dataset and learn how to deal with words that are not present in this vocabulary when working with other text sources. Aside from this you will also learn how to:
 
- read text files
- work with defaultdict
- work with string data

In [15]:
import string
from collections import defaultdict

### Read Text Data

A tagged dataset taken from the Wall Street Journal is provided in the file `WSJ_02-21.pos`. 

To read this file you can use Python's context manager by using the `with` keyword and specifying the name of the file you wish to read. To actually save the contents of the file into memory you will need to use the `readlines()` method and store its return value in a variable. 

Python's context managers are great because you don't need to explicitly close the connection to the file, this is done under the hood:

In [1]:
# Read lines from 'WSJ_02-21.pos' file and save them into the 'lines' variable
with open("./data/WSJ_02-21.pos", 'r') as f:
    lines = f.readlines()

In [6]:
# Print columns for reference
print("\t\tWord", "\tTag\n")

# Print first five lines of the dataset
for i in range(5):
    print(f'line number {i+1}: {lines[i]}')

line number 1: In	IN

line number 2: an	DT

line number 3: Oct.	NNP

line number 4: 19	CD

line number 5: review	NN



Each line within the dataset has a word followed by its corresponding tag. However since the printing was done using a formatted string it can be inferred that the **word** and the **tag** are separated by a tab (or some spaces) and there is a newline at the end of each line (notice that there is a space between each line). 

If you want to understand the meaning of these tags you can take a look [here](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html).

To better understand how the information is structured in the dataset it is recommended to print an unformatted version of it:

In [3]:
# Print first line (unformatted)
lines[0]

'In\tIN\n'

Indeed there is a tab between the word and the tag and a newline at the end of each line.

### Creating a vocabulary

Now that you understand how the dataset is structured, you will create a vocabulary out of it. A vocabulary is made up of every word that appeared at least 2 times in the dataset. 
For this, follow these steps:
- Get only the words from the dataset
- Use a defaultdict to count the number of times each word appears
- Filter the dict to only include words that appeared at least 2 times
- Create a list out of the filtered dict
- Sort the list

For step 1 you can use the fact that every word and tag are separated by a tab and that words always come first. Using list comprehension the words list can be created like this:

In [13]:
# Get the words from each line in the dataset
words = [line.split('\t')[0] for line in lines]

Step 2 can be done easily by leveraging `defaultdict`. In case you aren't familiar with defaultdicts they are a special kind of dictionaries that **return the "zero" value of a type if you try to access a key that does not exist**. Since you want the frequencies of words, you should define the defaultdict with a type of `int`. 

Now you don't need to worry about the case when the word is not present within the dictionary because getting the value for that key will simply return a zero. Isn't that cool?

In [16]:
# Define defaultdict of type 'int'
freq = defaultdict(int)

# Count frequency of ocurrence for each word in the dataset
for word in words:
    freq[word] += 1

Filtering the `freq` dictionary can be done using list comprehensions again (aren't they handy?). You should filter out words that appeared only once and also words that are just a newline character:

In [17]:
# Create the vocabulary by filtering the 'freq' dictionary
vocab = [k for k, v in freq.items() if (v > 1 and v != '\n')]

Finally, the `sort` method will take care of the final step. Notice that it changes the list directly so you don't need to reassign the `vocab` variable:

In [18]:
# Sort the vocabulary
vocab.sort()

In [19]:
# Print some random values of the vocabulary
for i in range(4000, 4005):
    print(vocab[i])

Earlier
Early
Earnings
Earth
Earthquake


In [20]:
# write the vocabulary into a txt file
with open('vocabulary.txt', 'w') as f:
    for word in vocab:
        f.write(word + '\n')

## Processing new text sources

### Dealing with unknown words

Now that you have a vocabulary, you will use it when processing new text sources. **A new text will have words that do not appear in the current vocabulary**. To tackle this, you can simply classify each new word as an unknown one, but you can do better by creating a function that tries to classify the type of each unknown word and assign it a corresponding `unknown token`. 

This function will do the following checks and return an appropriate token:

   - Check if the unknown word contains any character that is a digit 
       - return `--unk_digit--`
   - Check if the unknown word contains any punctuation character 
       - return `--unk_punct--`
   - Check if the unknown word contains any upper-case character 
       - return `--unk_upper--`
   - Check if the unknown word ends with a suffix that could indicate it is a noun, verb, adjective or adverb 
        - return `--unk_noun--`, `--unk_verb--`, `--unk_adj--`, `--unk_adv--` respectively

If a word fails to fall under any condition then its token will be a plain `--unk--`. The conditions will be evaluated in the same order as listed here. So if a word contains a punctuation character but does not contain digits, it will fall under the second condition. To achieve this behaviour some if/elif statements can be used along with early returns. 

This function is implemented next. Notice that the `any()` function is being heavily used. It returns `True` if at least one of the cases it evaluates is `True`.

In [33]:
def assign_unkown(word):
    """
    This function assign a token to an unknown word.
    
    Parameters:
    ------------
    word: string
        word to be assigned if unkown
    
    Returns:
    --------
    token: string
        token it is assigned to
    """

    # Punctuation characters
    # Try printing them out in a new cell!
    punct = set(string.punctuation)

    # Suffixes
    noun_suffix = ["action", "age", "ance", "cy", "dom", "ee", "ence", "er", "hood", "ion", "ism", "ist", "ity", "ling", "ment", "ness", "or", "ry", "scape", "ship", "ty"]
    verb_suffix = ["ate", "ify", "ise", "ize"]
    adj_suffix = ["able", "ese", "ful", "i", "ian", "ible", "ic", "ish", "ive", "less", "ly", "ous"]
    adv_suffix = ["ward", "wards", "wise"]

     # Loop the characters in the word, check if any is a digit
    if any(char.isdigit() for char in word):
        token = '--unk_digit--'
    
    # Loop the characters in the word, check if any is a punctuation character
    elif any(char in punct for char in word):
        token = '--unk_punct--'
    
    # Loop the characters in the word, check if any is an upper case character
    elif any(char.isupper() for char in word):
        token = '--unk_upper--'
    
    # Check if word ends with any noun suffix
    elif any(word.endswith(suffix) for suffix in noun_suffix):
        token = '--unk_noun--'
    
    # Check if word ends with any verb suffix
    elif any(word.endswith(suffix) for suffix in verb_suffix):
        token = '--unk_verb--'
    
    # Check if word ends with any adjective suffix
    elif any(word.endswith(suffix) for suffix in adj_suffix):
        token = '--unk_adj--'
    
    # Check if word ends with any adverb suffix
    elif any(word.endswith(suffix) for suffix in adv_suffix):
        token = '--unk_adv--'
    
    else:
        token = '--unk--'
    
    return token


A POS tagger will always encounter words that are not within the vocabulary that is being used. By augmenting the dataset to include these `unknown word tokens` you are helping the tagger to have a better idea of the appropriate tag for these words. 

### Getting the correct tag for a word

All that is left is to implement a function that will get the correct tag for a particular word taking special considerations for unknown words. Since the dataset provides each word and tag within the same line and a word being known depends on the vocabulary used, these two elements should be arguments to this function.

This function should check if a line is empty and if so, it should return a placeholder word and tag, `--n--` and `--s--` respectively. 

If not, it should process the line to return the correct word and tag pair, considering if a word is unknown in which scenario the function `assign_unk()` should be used.

The function is implemented next. Notice that the `split()` method can be used without specifying the delimiter, in which case it will default to any whitespace.

In [29]:
def get_word_tag(line, vocab):
    """
    This function returns a word and tag pair for a given input line

    Parameters:
    -----------
    line: string
        input line
    vocab: list
        vocabulary list
    
    Returns:
    --------
    a (word, tag) tuple

    """

    # If line is empty return placeholders for word and tag
    if not line.split():
        word = '--n--'
        tag = '--s--'
    
    else:
        # Split line to separate word and tag
        word, tag = line.split()

        # Check if word is not in vocabulary
        if word not in vocab: 
            # Handle unknown word
            tag = assign_unkown(word)
    
    return word, tag


Now you can try this function with some examples to test that it is working as intended:

In [30]:
get_word_tag('\n', vocab)

('--n--', '--s--')

Since this line only includes a newline character it returns a placeholder word and tag.

In [31]:
get_word_tag('In\tIN\n', vocab)

('In', 'IN')

This one is a valid line and the function does a fair job at returning the correct (word, tag) pair.

In [34]:
get_word_tag('tardigrade\tNN\n', vocab)

('tardigrade', '--unk--')

This line includes a noun that is not present in the vocabulary. 

The `assign_unk` function fails to detect that it is a noun so it returns an `unknown token`.

In [35]:
get_word_tag('scrutinize\tVB\n', vocab)

('scrutinize', '--unk_verb--')

This line includes a verb that is not present in the vocabulary. 

In this case the `assign_unk` is able to detect that it is a verb so it returns an `unknown verb token`.

# Parts-of-Speech Tagging - Working with tags and Numpy

In this lecture notebook you will create a matrix using some tag information and then modify it using different approaches.
This will serve as hands-on experience working with Numpy and as an introduction to some elements used for POS tagging.

In [55]:
import numpy as np
import pandas as pd
from itertools import product
import math

### Some information on tags

For this notebook you will be using a toy example including only three tags (or states). In a real world application there are many more tags which can be found [here](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html).

In [37]:
# Define tags for Adverb, Noun and To (the preposition) , respectively
tags = ['RB', 'NN', 'TO']

In this week's assignment you will construct some dictionaries that provide useful information of the tags and words you will be working with. 

One of these dictionaries is the `transition_counts` which counts the number of times a particular tag happened next to another. The keys of this dictionary have the form `(previous_tag, tag)` and the values are the frequency of occurrences.

Another one is the `emission_counts` dictionary which will count the number of times a particular pair of `(tag, word)` appeared in the training dataset.

In general think of `transition` when working with tags only and of `emission` when working with tags and words.

In this notebook you will be looking at the first one:

In [38]:
# Define 'transition_counts' dictionary
# Note: values are the same as the ones in the assignment
transition_counts = {
    ('NN', 'NN'): 16241,
    ('RB', 'RB'): 2263,
    ('TO', 'TO'): 2,
    ('NN', 'TO'): 5256,
    ('RB', 'TO'): 855,
    ('TO', 'NN'): 734,
    ('NN', 'RB'): 2431,
    ('RB', 'NN'): 358,
    ('TO', 'RB'): 200
}

Notice that there are 9 combinations of the 3 tags used. Each tag can appear after the same tag so you should include those as well.

### Using Numpy for matrix creation

Now you will create a matrix that includes these frequencies using Numpy arrays:

In [40]:
# Store the number of tags in the 'num_tags' variable
num_tags = len(tags)

# Initialize a 3X3 numpy array with zeros
transition_matrix = np.zeros((num_tags, num_tags))

# print the matrix
print(transition_matrix)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


Visually you can see the matrix has the correct dimensions. Don't forget you can check this too using the `shape` attribute:

In [41]:
# Print shape of the matrix
transition_matrix.shape

(3, 3)

Before filling this matrix with the values of the `transition_counts` dictionary you should sort the tags so that their placement in the matrix is consistent:

In [42]:
# Create sorted version of the tag's list
sorted_tags = sorted(tags)

print(sorted_tags)

['NN', 'RB', 'TO']


To fill this matrix with the correct values you can use a `double for loop`. You could also use `itertools.product` to one line this double loop:

In [43]:
# Loop rows
for i in range(num_tags):
    for j in range(num_tags):
        tags_tuple = (sorted_tags[i], sorted_tags[j])
        transition_matrix[i, j] = transition_counts.get(tags_tuple)

print(transition_matrix)

[[1.6241e+04 2.4310e+03 5.2560e+03]
 [3.5800e+02 2.2630e+03 8.5500e+02]
 [7.3400e+02 2.0000e+02 2.0000e+00]]


Looks like this worked fine. However the matrix can be hard to read as `Numpy` is more about efficiency, rather than presenting values in a pretty format. 

For this you can use a `Pandas DataFrame`. In particular, a function that takes the matrix as input and prints out a pretty version of it will be very useful:

In [51]:
# Define 'print_matrix' function
def print_matrix(matrix):
    print(pd.DataFrame(matrix, index=sorted_tags, columns=sorted_tags))

In [52]:
# Print the 'transition_matrix' by calling the 'print_matrix' function
print_matrix(transition_matrix)

         NN      RB      TO
NN  16241.0  2431.0  5256.0
RB    358.0  2263.0   855.0
TO    734.0   200.0     2.0


### Working with Numpy for matrix manipulation

Now that you got the matrix set up it is time to see how a matrix can be manipulated after being created. 

`Numpy` allows vectorized operations which means that operations that would normally include looping over the matrix can be done in a simpler manner. This is consistent with treating numpy arrays as matrices since you get support for common matrix operations. You can do matrix multiplication, scalar multiplication, vector addition and many more!

A trickier example is to normalize each row so that each value is equal to $\frac{value}{sum \,of \,row}$.

This can be easily done with vectorization.

In [53]:
# Compute sum of row for each row
row_sum = transition_matrix.sum(axis = 1, keepdims=True)

# Normalize transition matrix
transition_matrix = transition_matrix/row_sum

# Print normalized matrix
print_matrix(transition_matrix)

          NN        RB        TO
NN  0.678745  0.101596  0.219659
RB  0.102992  0.651036  0.245972
TO  0.784188  0.213675  0.002137


In [54]:
transition_matrix.sum(axis = 1, keepdims=True)

array([[1.],
       [1.],
       [1.]])

For a final example you are asked to modify each value of the diagonal of the matrix so that they are equal to the `log` of the sum of the current row plus the current value. When doing mathematical operations like this one don't forget to import the `math` module. 

This can be done using a standard `for loop` or `vectorization`. You'll see both in action:

In [74]:
t_matrix_for = np.copy(transition_matrix)
t_matrix_np = np.copy(transition_matrix)

#### Using a for-loop

In [75]:
for i in range(num_tags):
    t_matrix_for[i, i] = t_matrix_for[i, i] + math.log(row_sum[i, 0])

# Print matrix
print_matrix(t_matrix_for)

           NN        RB        TO
NN  10.761549  0.101596  0.219659
RB   0.102992  8.804673  0.245972
TO   0.784188  0.213675  6.843752


#### Using vectorization

In [76]:
d = np.diag(t_matrix_np).reshape((3, 1))

Now that the diagonal has the correct shape you can do the vectorized operation by applying the `math.log()` function to the `rows_sum` array and adding the diagonal. 

To apply a function to each element of a numpy array use Numpy's `vectorize()` function providing the desired function as a parameter. This function returns a vectorized function that accepts a numpy array as a parameter. 

To update the original matrix you can use Numpy's `fill_diagonal()` function.

In [77]:
d = d + np.vectorize(math.log)(row_sum)
np.fill_diagonal(t_matrix_np, d)
print(t_matrix_np)

[[10.76154917  0.10159646  0.21965898]
 [ 0.10299194  8.80467316  0.24597238]
 [ 0.78418803  0.21367521  6.84375223]]


In [78]:
# Check for equality
t_matrix_for == t_matrix_np

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True]])